In [8]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [9]:
import torch
import numpy as np

from utils.model import build_resnet50
from utils.dataloaders import create_dataloaders
from utils.checkpoint import load_checkpoint
from utils.metrics import evaluate_model

In [10]:
if torch.backends.mps.is_available():
    device = torch.device("mps")

elif torch.cuda.is_available():
    device = torch.device("cuda")

else:
    device = torch.device("cpu")

print(device)

mps


In [11]:
dataset_path = "../datasets/COVID-19_Radiography_Dataset"

(
    train_loader,
    val_loader,
    test_loader,
    train_df,
    val_df,
    test_df
) = create_dataloaders(
    dataset_path=dataset_path,
    batch_size=32
)

print(f"Test Images : {len(test_df)}")

Test Images : 3175


In [12]:
model = build_resnet50()

model = model.to(device)

print(model.fc)

Linear(in_features=2048, out_features=4, bias=True)


In [13]:
model = load_checkpoint(
    model,
    "../checkpoints/best_resnet50.pth",
    device
)

model.eval()

print("Best model loaded successfully!")

Best model loaded successfully!


In [14]:
results, y_true, y_pred, y_prob = evaluate_model(
    model=model,
    dataloader=test_loader,
    device=device
)

In [15]:
print("=" * 50)

print(f"Accuracy : {results['Accuracy']:.4f}")

print(f"Precision : {results['Precision']:.4f}")

print(f"Recall : {results['Recall']:.4f}")

print(f"F1 Score : {results['F1 Score']:.4f}")

print(f"AUC : {results['AUC']:.4f}")

print("=" * 50)

Accuracy : 0.9622
Precision : 0.9622
Recall : 0.9622
F1 Score : 0.9622
AUC : 0.9919


In [16]:
print(results["Classification Report"])

              precision    recall  f1-score   support

           0     0.9815    0.9779    0.9797       542
           1     0.9622    0.9653    0.9638      1529
           2     0.9476    0.9424    0.9450       902
           3     0.9755    0.9851    0.9803       202

    accuracy                         0.9622      3175
   macro avg     0.9667    0.9677    0.9672      3175
weighted avg     0.9622    0.9622    0.9622      3175



In [17]:
print(results["Confusion Matrix"])

[[ 530    6    4    2]
 [   7 1476   43    3]
 [   2   50  850    0]
 [   1    2    0  199]]
